# SilentVoice ISL Recognition - Google Colab Training

This notebook orchestrates the complete ML pipeline for training the SilentVoice ISL recognition model using the INCLUDE dataset.

## Pipeline Stages:
1. **Environment Setup** - Install dependencies and mount Google Drive
2. **Dataset Preparation** - Upload and organize INCLUDE dataset
3. **Preprocessing** - Extract landmarks and generate training sequences
4. **Training** - Train BiLSTM model with checkpointing
5. **Evaluation** - Evaluate model on test set
6. **Checkpoint Management** - Save best model to Google Drive

## Instructions:
- Upload INCLUDE dataset to Google Drive: `My Drive/SilentVoice/datasets/INCLUDE/`
- Checkpoints will be saved to: `My Drive/SilentVoice/models/checkpoints/`
- Preprocessed data will be saved to: `My Drive/SilentVoice/datasets/processed/`

## 1. Environment Setup

In [ ]:
# Install dependencies
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install opencv-python mediapipe numpy pandas pyyaml scikit-learn
!pip install matplotlib seaborn tensorboard
!pip install pydantic python-multipart uvicorn fastapi

print("Dependencies installed successfully!")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("Google Drive mounted successfully!")

In [ ]:
# Set up paths
import os
from pathlib import Path

# Google Drive paths
DRIVE_ROOT = Path('/content/drive/MyDrive/SilentVoice')
DATASET_ROOT = DRIVE_ROOT / 'datasets' / 'INCLUDE'
PROCESSED_ROOT = DRIVE_ROOT / 'datasets' / 'processed'
CHECKPOINT_DIR = DRIVE_ROOT / 'models' / 'checkpoints'
LOG_DIR = DRIVE_ROOT / 'models' / 'logs'

# Create directories
for path in [DATASET_ROOT, PROCESSED_ROOT, CHECKPOINT_DIR, LOG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Drive root: {DRIVE_ROOT}")
print(f"Dataset root: {DATASET_ROOT}")
print(f"Processed root: {PROCESSED_ROOT}")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")
print(f"Log dir: {LOG_DIR}")

## 2. Upload Project Modules

Upload the required ML modules from the SilentVoice project to Colab.

In [ ]:
# Create project structure in Colab
import shutil

# Create ml directory structure
ml_dirs = [
    'ml/models',
    'ml/training',
    'ml/datasets',
    'ml/inference/processors',
    'ml/utils',
    'ml/evaluation',
    'ml/translation',
]

for dir_path in ml_dirs:
    Path(dir_path).mkdir(parents=True, exist_ok=True)
    # Create __init__.py
    (Path(dir_path) / '__init__.py').touch()

print("ML directory structure created!")

### Upload Required Files

**Manual Step**: Upload the following files from your local SilentVoice project to Colab:

1. **Model files**:
   - `ml/models/bilstm_baseline.py`
   - `ml/models/__init__.py`

2. **Training files**:
   - `ml/training/trainer.py`
   - `ml/training/early_stopping.py`
   - `ml/training/checkpoint.py`
   - `ml/training/__init__.py`

3. **Dataset files**:
   - `ml/datasets/dataloader.py`
   - `ml/datasets/preprocessing_pipeline.py`
   - `ml/datasets/preprocess_include.py`
   - `ml/datasets/sequence_generator.py`
   - `ml/datasets/metadata_manager.py`
   - `ml/datasets/dataset_validator.py`
   - `ml/datasets/video_preprocessor.py`
   - `ml/datasets/frame_extractor.py`
   - `ml/datasets/quality_filter.py`
   - `ml/datasets/__init__.py`

4. **Inference processor files**:
   - `ml/inference/processors/landmark_extractor.py`
   - `ml/inference/processors/sequence_builder.py`
   - `ml/inference/processors/data_augmentation.py`
   - `ml/inference/processors/__init__.py`

5. **Utility files**:
   - `ml/utils/file_helpers.py`
   - `ml/utils/seed.py`
   - `ml/utils/__init__.py`

6. **Evaluation files**:
   - `ml/evaluation/metrics.py`
   - `ml/evaluation/visualization.py`
   - `ml/evaluation/__init__.py`

7. **Translation files**:
   - `ml/translation/translator.py`
   - `ml/translation/label_mapping.py`
   - `ml/translation/prediction.py`
   - `ml/translation/__init__.py`

8. **Main scripts**:
   - `ml/train.py`
   - `ml/evaluate.py`

**Use the file upload button in Colab or drag and drop these files to the appropriate directories.**

In [ ]:
# Verify file uploads
import os

required_files = [
    'ml/models/bilstm_baseline.py',
    'ml/training/trainer.py',
    'ml/datasets/dataloader.py',
    'ml/datasets/preprocessing_pipeline.py',
    'ml/datasets/preprocess_include.py',
    'ml/inference/processors/landmark_extractor.py',
    'ml/utils/file_helpers.py',
    'ml/train.py',
]

missing_files = []
for file_path in required_files:
    if not Path(file_path).exists():
        missing_files.append(file_path)

if missing_files:
    print("Missing files:")
    for file in missing_files:
        print(f"  - {file}")
    print("\nPlease upload the missing files before proceeding.")
else:
    print("All required files are present!")

## 3. Dataset Preparation

Upload your INCLUDE dataset to Google Drive at: `My Drive/SilentVoice/datasets/INCLUDE/`

Expected structure:
```
My Drive/SilentVoice/datasets/INCLUDE/
├── videos/
│   ├── subject_01/
│   │   ├── gesture_01.mp4
│   │   └── ...
│   └── ...
└── metadata.csv
```

In [ ]:
# Check if INCLUDE dataset exists
if DATASET_ROOT.exists():
    video_count = len(list(DATASET_ROOT.rglob('*.mp4')))
    video_count += len(list(DATASET_ROOT.rglob('*.avi')))
    video_count += len(list(DATASET_ROOT.rglob('*.mov')))
    
    metadata_file = DATASET_ROOT / 'metadata.csv'
    
    print(f"INCLUDE dataset found at: {DATASET_ROOT}")
    print(f"Video files found: {video_count}")
    print(f"Metadata file exists: {metadata_file.exists()}")
    
    if not metadata_file.exists():
        print("\nWARNING: metadata.csv not found. Will create sample metadata.")
else:
    print(f"INCLUDE dataset NOT found at: {DATASET_ROOT}")
    print("Please upload the INCLUDE dataset to Google Drive first.")
    print("Expected path: My Drive/SilentVoice/datasets/INCLUDE/")

In [ ]:
# Create metadata file if it doesn't exist
if not (DATASET_ROOT / 'metadata.csv').exists():
    print("Creating sample metadata file...")
    
    # This will use the preprocess_include.py script to create metadata
    import sys
    sys.path.insert(0, '/content')
    
    from ml.datasets.preprocess_include import create_sample_metadata
    
    metadata_path = DRIVE_ROOT / 'datasets' / 'metadata.csv'
    success = create_sample_metadata(str(DATASET_ROOT), str(metadata_path))
    
    if success:
        print(f"Metadata created at: {metadata_path}")
        print("Please review and edit the metadata file if needed.")
    else:
        print("Failed to create metadata file.")

## 4. Preprocessing

Run the preprocessing pipeline to extract MediaPipe landmarks and generate training sequences.

In [ ]:
# Run preprocessing pipeline
import sys
sys.path.insert(0, '/content')

from ml.datasets.preprocess_include import preprocess_include_dataset

# Preprocessing configuration
preprocessing_success = preprocess_include_dataset(
    dataset_root=str(DATASET_ROOT),
    output_root=str(PROCESSED_ROOT),
    metadata_path=str(DRIVE_ROOT / 'datasets' / 'metadata.csv'),
    sequence_length=30,
    stride=15,
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15,
    apply_augmentation=True,
    skip_existing=False  # Set to True to skip already processed videos
)

if preprocessing_success:
    print("\nPreprocessing completed successfully!")
else:
    print("\nPreprocessing failed. Check the error messages above.")

In [ ]:
# Verify preprocessing output
import json

stats_file = PROCESSED_ROOT / 'pipeline_statistics.json'
if stats_file.exists():
    with open(stats_file, 'r') as f:
        stats = json.load(f)
    
    print("Preprocessing Statistics:")
    print(f"  Total videos: {stats.get('total_videos', 0)}")
    print(f"  Processed videos: {stats.get('processed_videos', 0)}")
    print(f"  Failed videos: {stats.get('failed_videos', 0)}")
    print(f"  Total sequences: {stats.get('total_sequences', 0)}")
    print(f"  Train sequences: {stats.get('train_sequences', 0)}")
    print(f"  Val sequences: {stats.get('val_sequences', 0)}")
    print(f"  Test sequences: {stats.get('test_sequences', 0)}")
else:
    print("Statistics file not found. Preprocessing may have failed.")

## 5. Training

Train the BiLSTM model on the preprocessed INCLUDE dataset.

In [ ]:
# Check GPU availability
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Run training
import sys
sys.path.insert(0, '/content')

# Training configuration
training_args = [
    '--data_dir', str(PROCESSED_ROOT),
    '--train_split', 'train',
    '--val_split', 'val',
    '--input_dim', '279',
    '--hidden_dim', '128',
    '--num_layers', '2',
    '--dropout', '0.3',
    '--bidirectional',
    '--pooling_type', 'mean',
    '--epochs', '50',
    '--batch_size', '32',
    '--learning_rate', '1e-3',
    '--weight_decay', '1e-4',
    '--patience', '10',
    '--scheduler', 'cosine',
    '--scheduler_t_max', '50',
    '--device', str(device),
    '--use_amp',
    '--num_workers', '4',
    '--checkpoint_dir', str(CHECKPOINT_DIR),
    '--log_dir', str(LOG_DIR),
    '--log_level', 'INFO'
]

# Import and run training
from ml.train import main as train_main
import argparse

# Create args namespace
args = argparse.Namespace()
args.data_dir = str(PROCESSED_ROOT)
args.train_split = 'train'
args.val_split = 'val'
args.input_dim = 279
args.hidden_dim = 128
args.num_layers = 2
args.num_classes = 25  # Will be auto-detected from dataset
args.dropout = 0.3
args.bidirectional = True
args.pooling_type = 'mean'
args.epochs = 50
args.batch_size = 32
args.learning_rate = 1e-3
args.weight_decay = 1e-4
args.patience = 10
args.seed = 42
args.scheduler = 'cosine'
args.scheduler_t_max = 50
args.scheduler_step_size = 10
args.scheduler_gamma = 0.1
args.device = str(device)
args.num_workers = 4
args.use_amp = True
args.deterministic = False
args.checkpoint_dir = str(CHECKPOINT_DIR)
args.log_dir = str(LOG_DIR)
args.resume_from = None
args.log_level = 'INFO'

# Run training
print("Starting training...")
train_main()

## 6. Evaluation

Evaluate the trained model on the test set.

In [ ]:
# Run evaluation
import sys
sys.path.insert(0, '/content')

# Evaluation configuration
best_model_path = CHECKPOINT_DIR / 'best_model.pt'

if best_model_path.exists():
    print(f"Evaluating model: {best_model_path}")
    
    from ml.evaluate import main as eval_main
    import argparse
    
    # Create args namespace
    args = argparse.Namespace()
    args.checkpoint = str(best_model_path)
    args.data_dir = str(PROCESSED_ROOT)
    args.test_split = 'test'
    args.input_dim = 279
    args.hidden_dim = 128
    args.num_layers = 2
    args.num_classes = 25
    args.dropout = 0.3
    args.bidirectional = True
    args.pooling_type = 'mean'
    args.batch_size = 32
    args.device = str(device)
    args.output_dir = str(DRIVE_ROOT / 'models' / 'evaluation')
    args.log_level = 'INFO'
    
    # Run evaluation
    print("Starting evaluation...")
    eval_main()
else:
    print(f"Best model not found at: {best_model_path}")
    print("Training may have failed or not completed.")

## 7. Checkpoint Management

Verify and manage saved checkpoints.

In [ ]:
# List saved checkpoints
import os
from pathlib import Path

checkpoint_files = list(CHECKPOINT_DIR.glob('*.pt'))

if checkpoint_files:
    print("Saved checkpoints:")
    for checkpoint_file in sorted(checkpoint_files):
        size_mb = checkpoint_file.stat().st_size / (1024 * 1024)
        print(f"  {checkpoint_file.name}: {size_mb:.2f} MB")
else:
    print("No checkpoints found.")

In [ ]:
# Download best model
from google.colab import files

best_model_path = CHECKPOINT_DIR / 'best_model.pt'

if best_model_path.exists():
    print(f"Downloading best model: {best_model_path}")
    files.download(str(best_model_path))
else:
    print(f"Best model not found at: {best_model_path}")

In [ ]:
# Download training logs
log_file = DRIVE_ROOT / 'models' / 'training.log'

if log_file.exists():
    print(f"Downloading training log: {log_file}")
    files.download(str(log_file))
else:
    print(f"Training log not found at: {log_file}")

## 8. Summary

### Training Pipeline Complete!

**Files saved to Google Drive:**
- Checkpoints: `My Drive/SilentVoice/models/checkpoints/`
- Logs: `My Drive/SilentVoice/models/logs/`
- Preprocessed data: `My Drive/SilentVoice/datasets/processed/`
- Evaluation results: `My Drive/SilentVoice/models/evaluation/`

**Next Steps:**
1. Download `best_model.pt` to your local project
2. Place it in `ml/models/checkpoints/best_model.pt`
3. Start the FastAPI backend: `python api/app.py`
4. Start the frontend: `cd web && npm run dev`
5. Test the model with real webcam input

**Model Information:**
- Architecture: BiLSTM with attention pooling
- Input: 279-dimensional MediaPipe landmarks
- Sequence length: 30 frames
- Classes: Auto-detected from dataset
- Training epochs: Completed with early stopping
- Best validation accuracy: Check training logs